<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2021%20-%20Optimization%20I%20-%20Linear%20Programming/lab9_LP_Domaine_Paul_Autard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9 Prescriptive Analytics - Optimization: Solving Resource Allocation Problems at Domaine Paul Autard using Linear Programming

## Objectives:

In this module, we will learn how to formulate a basic linear optimization model, for a resource allocation problem: Domaine Paul Autard. In particular, we will learn how to:

*   translate a problem description into a mathematical (algebraic) model
*   embed this model into a python optimization framework
*   choose the correct solution algorithm
*   solve and interpret the model
*   conduct sensitivity analysis

## Problem Description

The **3 types** of grapes harvested, and their quantities available, are as follows:

| Grenache | Syrah | Counoise |
|  --- | --- | --- |
| 1,700 cases | 1,200 cases | 500 cases

The domaine produces two varieties of red wine, termed "La Cote Ronde" (LCR) and "Domaine Paul Autard" (DPA). Each of them use the input ressources (the three types of grapes) differently.

| Output\Input | Grenache | Syrah | Counoise |
| --- |  --- | --- | --- |
1 case of **LCR wine**: | 0.5 cases | 0.5 cases | 0 cases
1 case of **DPA wine**: | 0.6 cases | 0.2 cases | 0.2 cases

Assume there is infinite demand for the wines, at the following retail prices:

| Wines | Retail Price |
| --- |  --- |
**LCR wine**: | 400 euros per case
**DPA wine**: | 600 euros per case

**How much of each wine should Paul Autard make?**

## Setup

Pyomo is the modeling package: it lets us write the model in Python. GLPK is the solver: the engine that actually finds the optimal plan. The `apt-get` line installs GLPK and works in **Google Colab**; it will not work on a Mac or on Windows.

In [ ]:
%%capture
# Install Pyomo and the GLPK solver, then load the packages.
!pip install pyomo
!apt-get install -y -qq glpk-utils
import pyomo.environ as pyo

## Optimization Model

We build the model in the same steps as the slides: decision variables, objective function, constraints, then solve.

To keep the numbers small, we measure wine in **hundreds of cases** and revenue in **thousands of euros**. For example, 100 cases of Cote Ronde sell for 400 × 100 = 40,000 euros, so its objective coefficient is 40; and 1,700 cases of Grenache becomes 17.

### Decision variables

Two decisions: how much Cote Ronde (`c`) and how much Domaine Paul Autard (`p`) to make. Neither can be negative.

In [ ]:
# Create the model and its two decision variables.
model = pyo.ConcreteModel()
model.c = pyo.Var(bounds=(0, None))   # Cote Ronde, in 100s of cases
model.p = pyo.Var(bounds=(0, None))   # Domaine Paul Autard, in 100s of cases

# Shorter names, so the formulas below read like the slides.
c = model.c
p = model.p

### Objective function

Maximize revenue: 40 per hundred cases of Cote Ronde, 60 per hundred cases of Domaine Paul Autard.

In [ ]:
# Revenue to maximize, in thousands of euros.
model.obj = pyo.Objective(expr=40*c + 60*p, sense=pyo.maximize)

### Constraints

One constraint per grape: the grapes the two wines use cannot exceed the grapes harvested. Each constraint gets a descriptive name, because that name is what appears in the sensitivity report later.

In [ ]:
# One constraint per grape (in 100s of cases).
model.GrenacheConstraint = pyo.Constraint(expr=0.5*c + 0.6*p <= 17)
model.SyrahConstraint = pyo.Constraint(expr=0.5*c + 0.2*p <= 12)
model.CounoiseConstraint = pyo.Constraint(expr=0.2*p <= 5)

In [ ]:
# Print the model to check it matches the formulation on the slides.
model.pprint()

### Solve the model

This is a linear program, so we use the GLPK solver. Two options are worth knowing:

* `opt.options['ranges'] = 'sens.txt'` asks GLPK to also write a **sensitivity report** to a file called `sens.txt`. We read it at the end of the lab.
* `symbolic_solver_labels=True` makes that report use our own names (`c`, `p`, `GrenacheConstraint`, ...). Without it, GLPK invents generic names such as `x2` and `c_u_x4_`.

In [ ]:
# Solve with GLPK and save the sensitivity report to sens.txt.
opt = pyo.SolverFactory('glpk')
opt.options['ranges'] = 'sens.txt'
results = opt.solve(model, symbolic_solver_labels=True)
print(results.solver.termination_condition)

In [ ]:
# Show the optimal production plan.
print("Optimal revenue (1000s of euros) =", model.obj())
print("Cote Ronde to make (100s of cases) =", c())
print("Domaine Paul Autard to make (100s of cases) =", p())

This is the mathematically guaranteed optimal solution, meaning, there does not exist any other production plan (any other combination of $c$ and $p$) that can generate higher revenues while satisfying the constraints of the problem.

### Optional: try to beat the solver

How confident can we be that there is no better plan out there? You do not need to check a plan by hand: the model already knows the revenue formula and the grape limits. The cell below copies the model (so the solver's answer stays untouched), plugs in your own plan, and lets the model report the revenue and whether each grape constraint holds.

**Experiment.** Change the two numbers (`5` and `13`) and re-run the cell. A big bonus goes to anyone who finds a plan that satisfies all three grape constraints and **strictly beats** the Pyomo solution (4, 25) on revenue.

In [ ]:
# Copy the model, plug in your own plan, and let the model score it.
my_model = model.clone()
my_model.c.value = 5    # Cote Ronde, 100s of cases
my_model.p.value = 13   # Domaine Paul Autard, 100s of cases

print("Solver revenue =", model.obj())
print("My revenue     =", my_model.obj())
for grape in [my_model.GrenacheConstraint, my_model.SyrahConstraint, my_model.CounoiseConstraint]:
    used = round(pyo.value(grape.body), 2)
    available = pyo.value(grape.upper)
    print(grape.name, ": uses", used, "of", available, "->", "OK" if used <= available else "NOT ENOUGH GRAPES")

## Sensitivity Analysis

The report has two pages. Page 1 has one row per **constraint**, page 2 one row per **decision variable**. The columns to read first:

* **Slack**: how much of a resource is left unused. A constraint with slack is not a bottleneck.
* **Marginal** (also called the **shadow price**): how much the optimal revenue rises if the constraint's right-hand side goes up by 1. A positive marginal marks a bottleneck.
* **Activity range**: how far the right-hand side can move before the marginal stops being valid.
* **Obj coef range** (page 2): how far a price can move before the optimal plan changes.

In [ ]:
# Print the sensitivity report GLPK wrote during the solve.
print(open('sens.txt').read())

### Follow up Questions and Exercises

**Question 1.** Which grapes are bottlenecks, and how can you tell from the report? How much Syrah goes unused?

**Answer**

*Provide your answer here*

**Question 2.** If the domaine could get 100 more cases of Grenache, how much would revenue go up? Would the same rate hold for 1,000 more cases?

**Answer**

*Provide your answer here*

**Exercise 1.** The "greedy" approach makes as many cases of the expensive wine (DPA) as the grapes allow, then makes Cote Ronde with what is left. Here it gives the same plan as Pyomo: (4, 25). Change the price of DPA from 600 to 450 euros (60 becomes 45 in the objective), re-build and re-solve the model. Is the greedy plan still optimal?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*